# Initialization

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim
import pyspark.sql.functions as F

#Reading from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# Transformations

- Trim text columns
- Separate product key column into category id and product key
- Normalize product line column
- Rename column headers
- Product cost has nulls but will leave as null since zero may be misleading

### Trimming text columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
        

### Parsing product key and category id

In [0]:
df = df.withColumn("category_id", F.regexp_replace(F.substring(F.col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key"))))

### Normalizing product line column

In [0]:
df = df.withColumn("prd_line",
               F.when(F.upper(col("prd_line")) == "R", "Road")
                 .when(F.upper(col("prd_line")) == "S", "Sport")
                 .when(F.upper(col("prd_line")) == "M", "Mountain")
                 .when(F.upper(col("prd_line")) == "T", "Touring")
                 .otherwise("N/A")
)

### Renaming column headers

In [0]:
COLUMN_NAMES = {
    "prd_id": "product_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date",
}

for old_name, new_name in COLUMN_NAMES.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity check dataframe

In [0]:
df.limit(10).display()

# Writing to silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

### Sanity check silver table

In [0]:
%sql

SELECT * FROM workspace.silver.crm_products LIMIT 10